# ASD risk-gene spatial enrichment in the adult brain (AHBA)

Regional enrichment of the 253-gene ASD rare-variant risk set across the Desikan-Killiany
atlas, the cortical surface map, and the spin test of its anterior-posterior alignment —
plus the sensitivity analyses that stress-test the posterior/occipital result.

> **Kernel:** the `spin` conda env (`Python (spin)`) — neuromaps / nilearn / statsmodels.
> Only `00_build_ahba_matrix.ipynb` needs the pinned `abagen` env.
> **Run from this directory** so `import spatial_helpers` resolves.

**Inputs** — the cached AHBA matrix and DK `.annot` files in `../data/` (notebook 00), and
the TADA exome table in `data/`.

**Outputs**
| | |
|---|---|
| `outputs/analysis/04_brain_spatial_enrichment/04_01_ASD_regional_enrichment.csv` | per-region enrichment, the map everything else reads |
| `04_01_ASD_matched_null.csv`, `04_01_ASD_dropocc_ap_*.csv` | sensitivity statistics |
| `outputs/figures/Fig1/Fig1G_ASD_{brain,legend,scatter,spin}.pdf` | Fig 1G elements, for assembly |
| `outputs/figures/supplement/04_01_ASD_sensitivity.pdf` | supplementary sensitivity figure |

The GO-topic (SYN / GR / MORPH) versions of everything here live in
`04_02_topic-spatial-enrichment.ipynb`; the two notebooks are independent.

## 0) Setup

In [ ]:
import os, glob, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import spatial_helpers as sh
warnings.filterwarnings("ignore")

PROJ = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))  # repo root (run from this directory)
RESULTS  = f"{PROJ}/outputs/analysis/04_brain_spatial_enrichment"
CACHE    = f"{RESULTS}/spin_cache"
FIG_MAIN = f"{PROJ}/outputs/figures/Fig1"          # ASD map is main-text panel Fig 1G
FIG_SUPP = f"{PROJ}/outputs/figures/supplement"
for d in (RESULTS, FIG_MAIN, FIG_SUPP):
    os.makedirs(d, exist_ok=True)

# pdf.fonttype 42 embeds TrueType, so figure text stays editable in Illustrator
plt.rcParams.update({"font.size": 7, "axes.titlesize": 8, "axes.labelsize": 7,
                     "xtick.labelsize": 6, "ytick.labelsize": 6,
                     "pdf.fonttype": 42, "ps.fonttype": 42, "mathtext.default": "regular",
                     "font.family": "sans-serif",
                     "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"]})
P_SPIN, P_PERM = r"$P_{\mathrm{spin}}$", r"$P_{\mathrm{perm}}$"

# 253-gene ASD risk set: TADA FDR <= 0.001, flagged genes excluded
tada = pd.read_csv(f"{PROJ}/data/full_results_wcounts_2025-10-08.txt", sep="\t")
asd_genes = tada[(tada.FDR < 0.001) & (~tada.FLAG)]["gene"].tolist()

expr, info = sh.load_ahba()
background = sh.brain_expressed_background(expr)
Z = sh.zscore_expression(expr, background)
dk = sh.dk_region_order()                       # 34 LH cortical regions, spin-test order

print(f"ASD risk genes : {len(asd_genes)} "
      f"({sum(g in background for g in asd_genes)} in the brain-expressed background)")
print(f"AHBA matrix    : {expr.shape[0]} regions x {expr.shape[1]} genes")
print(f"background     : {len(background)} genes")

## 1) Regional enrichment vs a brain-expressed background

Per gene, expression is z-scored across regions; the per-region set score is the mean z
across ASD genes. Significance comes from **10,000 size-matched gene sets resampled from
brain-expressed genes only** (above the 10th percentile of mean AHBA expression) — the
appropriate comparison for a set already known to be neuronal. `z_vs_bg` is the
background-corrected enrichment that every map and spin test downstream uses. Empirical
p-values are BH-FDR corrected across the region labels.

In [ ]:
enr = sh.regional_enrichment(Z, info, asd_genes, n_perm=10000, seed=0)
enr.to_csv(f"{RESULTS}/04_01_ASD_regional_enrichment.csv", index=False)

z_by_region = dict(zip(enr.region, enr.z_vs_bg))
asd_map = np.array([z_by_region.get(r, np.nan) for r in dk])   # 34 cortical regions, spin order
vmax = sh.cortical_vmax(enr)          # colour limit from cortex; subcortical |z| is larger

print(f"regions FDR<0.05: {(enr.fdr < 0.05).sum()}/{len(enr)}   "
      f"FDR<0.001: {(enr.fdr < 0.001).sum()}/{len(enr)}")
print(f"mean set-score   cortex {enr.loc[enr.structure == 'cortex', 'obs'].mean():.3f}"
      f"   subcortex/brainstem {enr.loc[enr.structure != 'cortex', 'obs'].mean():.3f}")
enr.head(8)

## 2) Spin test

The spin test controls for the strong spatial autocorrelation of cortical transcription
(Alexander-Bloch / Váša; Dear et al.'s `cornblath` parcel-spin at 41k density). Run it
against the map written above — on the cluster:

```
./submit_spin_tests.sh
```

or locally for the ASD map alone:

```
python run_spin_test.py --n-spin 5000 --prefix 04_01_ASD_ --asd-cols z_vs_bg
```

Three result tables are read back below:

- `04_01_ASD_spin_results_*` — correlation with the A–P axis and Dear's C1/C2/C3 gradients
- `04_01_ASD_cortex_perregion_*` — spin-corrected per-region enrichment (BH FDR `q`, plus
  max-statistic FWE `p_fwe`); the significant regions are outlined on the surface map
- `04_01_ASD_occipital_focal_*` — the occipital-cluster focal test

In [ ]:
latest = lambda pattern: sorted(glob.glob(pattern))[-1]
spin_corr = pd.read_csv(latest(f"{RESULTS}/04_01_ASD_spin_results_41k_*.csv"))
perregion = pd.read_csv(latest(f"{RESULTS}/04_01_ASD_cortex_perregion_41k_*.csv"))
occ_focal = pd.read_csv(latest(f"{RESULTS}/04_01_ASD_occipital_focal_41k_*.csv"))

# Regions outlined on the surface map. Switch to ("p_fwe", 0.05) for max-statistic
# family-wise control instead of BH-FDR across regions.
SIG_COL, SIG_CUT = "q", 0.05
sig_regions = set(perregion.loc[perregion[SIG_COL] < SIG_CUT, "C"])

print(spin_corr.to_string(index=False))
print()
print(occ_focal.to_string(index=False))
print(f"\n{len(sig_regions)} regions with {SIG_COL} < {SIG_CUT}: {sorted(sig_regions)}")

## 3) Main-text figure elements (Fig 1G)

Each panel is exported **separately** for manual assembly in Illustrator. The brain map
goes out as a raster embedded in a PDF — the 164k vector surface is too large to keep as
true vector — while the scatter, colorbar and spin panel are real vector PDFs.

Surface density is `spatial_helpers.DENSITY`, currently `"fsaverage"` (native 164k), which
keeps small regions like bankssts and the significance outlines crisp at ~7 s per panel.
**Set `sh.DENSITY = "fsaverage5"` (10k verts/hemi) for a fast draft** — the map is constant
within each DK region, so the coarse mesh looks near-identical and renders ~16x faster.

### 3.1 Cortical surface map + colorbar

In [ ]:
cmap = sh.diverging_cmap()                         # RdBu_r for the all-ASD map

fig = sh.plot_dk_surface(z_by_region, cmap, vmax, outline=sig_regions, bare=True)
fig.savefig(f"{FIG_MAIN}/Fig1G_ASD_map_clean.png", dpi=300, bbox_inches="tight", pad_inches=0)
plt.show()

# embed the raster in a small PDF so the assembled figure stays one vector document
img = plt.imread(f"{FIG_MAIN}/Fig1G_ASD_map_clean.png")
f = plt.figure(figsize=(2.2, 2.2 / (img.shape[1] / img.shape[0])))
f.add_axes([0, 0, 1, 1]).imshow(img); f.axes[0].axis("off")
f.savefig(f"{FIG_MAIN}/Fig1G_ASD_brain.pdf", dpi=300); plt.close(f)

leg = sh.colorbar_panel(cmap, vmax, label="Z")
leg.savefig(f"{FIG_MAIN}/Fig1G_ASD_legend.pdf", bbox_inches="tight")
leg.savefig(f"{FIG_MAIN}/Fig1G_ASD_legend.png", dpi=600, bbox_inches="tight"); plt.close(leg)
print(f"vmax = {vmax:.2f} | saved brain + legend to {FIG_MAIN}")

### 3.2 Anterior–posterior scatter

Regional enrichment against A–P position. Plotted **posterior-positive** (x = −Y) so the
reported ρ is positive for a posterior-leaning map, matching the spin panel below. This
scatter is main-text for the all-ASD set only; the topic maps use the spin panel alone.

In [ ]:
ap_pos = sh.ap_position()
ctx = enr[enr.structure == "cortex"].copy()
ctx["ap"] = -ctx.region.map(ap_pos)                       # posterior-positive
ctx = ctx.dropna(subset=["ap"])
rho_ap = -spin_corr.set_index("C").loc["AP", "enrich_z"]  # flip to match the x-axis

fig, ax = plt.subplots(figsize=(0.95, 0.9))               # near-square main-text panel
shade = ((ctx.z_vs_bg + vmax) / (2 * vmax)).clip(0, 1)
ax.scatter(ctx.ap, ctx.z_vs_bg, c=cmap(shade), s=5, edgecolor="#333", lw=0.25, zorder=3)
xs = np.linspace(ctx.ap.min(), ctx.ap.max(), 30)
ax.plot(xs, np.polyval(np.polyfit(ctx.ap, ctx.z_vs_bg, 1), xs), "--", color="#666", lw=0.6)
ax.set_xticks([])                                         # position is ordinal here, not mm
ax.set_xlabel("anterior → posterior", fontsize=5, labelpad=2)
ax.set_ylabel("enrichment Z", fontsize=5, labelpad=2)
ax.tick_params(axis="y", labelsize=4.5, length=1.5, pad=1, width=0.4)
ax.text(0.04, 0.96, f"ρ = {rho_ap:.2f}", transform=ax.transAxes, ha="left", va="top", fontsize=5)
for s in ("top", "right"): ax.spines[s].set_visible(False)
for s in ("left", "bottom"): ax.spines[s].set_linewidth(0.4)
fig.savefig(f"{FIG_MAIN}/Fig1G_ASD_scatter.pdf", bbox_inches="tight")
fig.savefig(f"{FIG_MAIN}/Fig1G_ASD_scatter.png", dpi=600, bbox_inches="tight")
plt.show()

### 3.3 A–P spin-null panel

The observed A–P correlation against its null: the ASD map correlated with 5,000 spun
copies of the A–P axis. Reusing the cached component nulls from the cluster run means no
re-spinning here. The reconstructed p matches the `p` column of the cluster CSV.

In [ ]:
# cached component nulls: (34 regions, 4 components, n_spin); AP is component index 3
spun_ap = np.load(latest(f"{CACHE}/nulls_41k_n*_seed0_4comp.npy"))[:, 3, :]
ap_null = -np.array([sh.masked_spearman(asd_map, spun_ap[:, i]) for i in range(spun_ap.shape[1])])

row = spin_corr.set_index("C").loc["AP"]
obs, p = -row["enrich_z"], row["p"]
print(f"A–P ρ = {obs:.3f} | cluster {P_SPIN} = {p:.4f} | reconstructed = {sh.emp_p(obs, ap_null):.4f}")

fig, ax = plt.subplots(figsize=(0.92, 1.2))
sh.plot_spin_null(ax, ap_null, obs, "#b2182b", xlim=(-0.85, 0.85),
                  title=f"ρ = {obs:.2f}{'*' if p < 0.05 else ''} : {P_SPIN} = {sh.pstr(p)}")
ax.set_xlabel("Spearman ρ \n(vs A–P axis)", fontsize=5)
ax.set_xticks([-0.5, 0, 0.5]); ax.tick_params(labelsize=4.5)
fig.tight_layout()
fig.savefig(f"{FIG_MAIN}/Fig1G_ASD_spin.pdf")
fig.savefig(f"{FIG_MAIN}/Fig1G_ASD_spin.png", dpi=300)
plt.show()

## 4) Sensitivity analyses

Four stress-tests of the posterior/occipital result, compiled into one supplementary
figure in §4.5:

- **A — map non-uniformity.** Is the map spatially structured at all? Spatial variance of
  the regional enrichment Z vs 20,000 random size-matched gene sets.
- **B — A–P gradient ± occipital.** Is the gradient just the four occipital regions?
- **C — gene-property-matched null.** Does the occipital signal survive matching the
  background on expression level, CDS length and constraint (LOEUF)?
- **D — drop-top-k.** Is the effect distributed across genes, or carried by a handful?

Leave-one-gene-out is not run: drop-top-k subsumes it (k = 1 is the leave-one-out case,
and it also rules out a small group, which leave-one-out cannot).

### 4.1 Map non-uniformity

In [ ]:
obs_z, obs_var, null_var = sh.map_nonuniformity(Z, info, asd_genes, n_perm=20000, seed=0)
p_nonuniform = (np.sum(null_var >= obs_var) + 1) / (len(null_var) + 1)
print(f"spatial variance of enrichment Z = {obs_var:.2f} vs null mean {null_var.mean():.2f}, "
      f"P_perm = {p_nonuniform:.4g}")

### 4.2 Gene-property-matched null

ASD risk genes are long, constrained and highly expressed — all properties that covary
with cortical expression topography, and none of which the brain-expressed background
controls for. Null sets here are matched on the joint distribution of expression level,
CDS length and LOEUF (tertiles on each, crossed into up to 27 strata). The unmatched null
is shown alongside so the shift attributable to gene properties is visible.

In [ ]:
cds = pd.read_csv(f"{PROJ}/data/gene_cds_length.csv")
gene_mean = expr.mean(axis=0)
covariates = (pd.DataFrame({"gene": background,
                            "expr": np.log10(gene_mean[background].values + 1e-6)})
              .merge(cds, left_on="gene", right_on="gene_name", how="left")
              .assign(cds=lambda d: np.log10(d.union_cds_length))
              .merge(tada[["gene", "LOEUF"]], on="gene", how="left")
              .dropna(subset=["expr", "cds", "LOEUF"]).set_index("gene"))

gene_occ = sh.occipital_gene_scores(Z, info)          # per-gene occipital score
mn = sh.matched_null_occipital(asd_genes, gene_occ, background, covariates, n_perm=5000, seed=0)

pd.DataFrame([{"gene_set": "ASD", "n": mn["n"], "occipital_obs": mn["obs"],
               "unmatched_mean": mn["null_u"].mean(), "p_unmatched": mn["p_u"],
               "matched_mean": mn["null_m"].mean(), "p_matched": mn["p_m"]}]) \
    .to_csv(f"{RESULTS}/04_01_ASD_matched_null.csv", index=False)
print(f"occipital enrichment {mn['obs']:.3f} | unmatched p = {mn['p_u']:.4f} | "
      f"matched p = {mn['p_m']:.4f} (matched null mean {mn['null_m'].mean():.3f})")

### 4.3 Drop-occipital A–P test

The A–P gradient and the occipital focal test are the same signal seen two ways. Removing
the four occipital regions and re-running the spin test asks whether the gradient is
broader than occipital. The spun A–P axis is generated once at 41k and cached, so the
matching test in notebook 02 reuses it. `N_SPIN = 1000` here for speed; raise to 5000 to
match the primary analysis.

In [ ]:
N_SPIN_DROPOCC = 1000
ap, ap_null = sh.cached_ap_spin_null(n_spin=N_SPIN_DROPOCC, density="41k", seed=0)
dropocc = sh.dropocc_ap_test(asd_map, ap, ap_null, dk)

pd.DataFrame([{"gene_set": "ASD", **dropocc}]) \
    .to_csv(f"{RESULTS}/04_01_ASD_dropocc_ap_41k_n{N_SPIN_DROPOCC}.csv", index=False)
print(f"full 34 regions : ρ = {-dropocc['rho_full']:+.3f}, {P_SPIN} = {dropocc['p_full']:.3f}")
print(f"occipital removed: ρ = {-dropocc['rho_dropocc']:+.3f}, {P_SPIN} = {dropocc['p_dropocc']:.3f}")

### 4.4 Drop-top-k

In [ ]:
ks, dtk, n_genes = sh.drop_top_k(asd_genes, gene_occ, max_k=30)
chance = float(gene_occ.mean())                       # expected score of a random background set
print(f"{n_genes} genes ranked by occipital score; enrichment {dtk[0]:.3f} (k=0) -> "
      f"{dtk[-1]:.3f} (k={ks[-1]}), chance = {chance:.3f}")

### 4.5 Supplementary figure

Journal figure width is 7.09 in; this panel is sized to it.

In [ ]:
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.colors import to_rgba

POS = "#b2182b"; ASD_COL = sh.ASD_COLOR
fig = plt.figure(figsize=(7.09, 4.7))
gs = GridSpec(2, 6, figure=fig, hspace=0.55, wspace=1.05)

# --- A1: distribution of regional enrichment Z across the 34 cortical regions ---
axA1 = fig.add_subplot(gs[0, 1:3])
axA1.hist(obs_z, bins=np.arange(-5, 6.5, 0.5), color=POS, alpha=0.75, edgecolor="white", lw=0.3)
axA1.axvline(0, color="0.6", lw=0.5, ls=":"); axA1.set_xlim(-5, 6.5)
axA1.set_xlabel("regional enrichment Z", fontsize=6); axA1.set_ylabel("# regions", fontsize=6)
axA1.set_title("ASD regional enrichment-Z", color=POS, fontsize=7, fontweight="bold")
axA1.tick_params(labelsize=5)
for s in ("top", "right"): axA1.spines[s].set_visible(False)

# --- A2: its spatial variance vs 20,000 random size-matched sets ---
axA2 = fig.add_subplot(gs[0, 3:5])
axA2.hist(null_var, bins=50, color="0.85", edgecolor="black", linewidth=0.05, density=True)
axA2.axvline(obs_var, color=POS, lw=2)
axA2.set_xlabel("spatial variance of enrichment Z", fontsize=6)
axA2.set_ylabel("density", fontsize=6); axA2.set_yticks([])
axA2.set_title(f"observed var = {obs_var:.1f},  {P_PERM} = {sh.pstr(p_nonuniform)}",
               fontsize=6.5, fontweight="bold")
axA2.tick_params(labelsize=5)
for s in ("top", "right", "left"): axA2.spines[s].set_visible(False)

# --- B: A-P gradient with and without the occipital regions ---
axB = fig.add_subplot(gs[1, 0:2])
xp = -ap
is_occ = np.array([r in sh.OCCIPITAL for r in dk])
keep = ~is_occ
axB.scatter(xp[keep], asd_map[keep], s=16, color=ASD_COL, edgecolor="#333", lw=0.3, zorder=3)
axB.scatter(xp[is_occ], asd_map[is_occ], s=34, facecolor=to_rgba(ASD_COL, 0.3),
            edgecolor=ASD_COL, lw=0.8, ls="--", zorder=4)
xs = np.linspace(xp.min(), xp.max(), 50)
axB.plot(xs, np.polyval(np.polyfit(xp, asd_map, 1), xs), "--", color="#aaa", lw=1)
axB.plot(xs, np.polyval(np.polyfit(xp[keep], asd_map[keep], 1), xs), "-", color=ASD_COL, lw=1.5)
axB.set_xticks([]); axB.set_xlabel(r"anterior $\rightarrow$ posterior"); axB.set_ylabel("enrichment Z")
axB.text(0.04, 0.97, f"full ρ={-dropocc['rho_full']:.2f} (p={dropocc['p_full']:.3f})\n"
                     f"−occ ρ={-dropocc['rho_dropocc']:.2f} (p={dropocc['p_dropocc']:.3f})",
         transform=axB.transAxes, fontsize=5, va="top")
axB.set_title("A–P gradient (± occipital)", fontsize=7, fontweight="bold")
for s in ("top", "right"): axB.spines[s].set_visible(False)

# --- C: gene-property-matched null ---
axC = fig.add_subplot(gs[1, 2:4])
axC.hist(mn["null_u"], bins=35, color="#cccccc", edgecolor="black", linewidth=0.05, alpha=.85)
axC.hist(mn["null_m"], bins=35, color="#7fb3d5", edgecolor="black", linewidth=0.05, alpha=.85)
axC.axvline(mn["obs"], color=POS, lw=1.5)
axC.set_xlabel("occipital enrichment"); axC.set_ylabel("null count")
axC.set_title(f"Gene-property-matched null\n(unmatched p={mn['p_u']:.4f}; matched p={mn['p_m']:.3f})",
              fontsize=6.5, fontweight="bold")
axC.legend(handles=[Patch(color="#cccccc"), Patch(color="#7fb3d5"), Line2D([0], [0], color=POS, lw=1.5)],
           labels=["unmatched", "matched", "observed"], fontsize=5, frameon=False, loc="upper left")
for s in ("top", "right"): axC.spines[s].set_visible(False)

# --- D: drop-top-k ---
axD = fig.add_subplot(gs[1, 4:6])
axD.plot(ks, dtk, color=ASD_COL, lw=1.6, label=f"ASD (n={n_genes})")
axD.axhline(chance, ls=":", color="#888", lw=1)
axD.text(0.99, chance, "chance", transform=axD.get_yaxis_transform(),
         ha="right", va="bottom", fontsize=5, color="#888")
axD.set_xlabel("top occipital genes removed (k)"); axD.set_ylabel("occipital enrichment (remaining)")
axD.set_title("Drop-top-k (ASD)", fontsize=7, fontweight="bold"); axD.legend(fontsize=5, frameon=False)
for s in ("top", "right"): axD.spines[s].set_visible(False)

for tag, ax in zip("ABCD", [axA1, axB, axC, axD]):
    ax.text(-0.28, 1.18, tag, transform=ax.transAxes, fontsize=10, fontweight="bold", va="top")

fig.savefig(f"{FIG_SUPP}/04_01_ASD_sensitivity.pdf", bbox_inches="tight")
fig.savefig(f"{FIG_SUPP}/04_01_ASD_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()